# 03 — Ablation analysis

Reproduce the ablation table (Sec. V-C, Table II) and visualise component
contributions as a waterfall chart.

All numbers below are reported in the paper. The cell at the end shows how to
compare them against your own ablation runs if you have them.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

FIG_DIR = ROOT / 'outputs/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Table II (paper) — AzSLD 50 unseen Top-1 (%)

| Configuration                                    | Top-1 |
|--------------------------------------------------|------:|
| Ours (Sapiens only, single template)             | 68.1  |
| Ours (Sapiens only, 5-template ensemble)         | 72.3  |
| Ours (full) — w/o prompt ensemble                | 75.4  |
| Ours (full) — w/o skeleton noise augmentation    | 77.1  |
| Ours (full) — fine-tuned visual encoders         | 74.2  |
| **Ours (full) — Sapiens + MotionBERT + 5-tpl**   | **79.8** |
| τ-init = 0.05                                    | 79.5  |
| τ-init = 0.07 (learned → ≈ 0.11)                 | 79.8  |
| τ-init = 0.20                                    | 79.0  |

In [ ]:
# Waterfall: starting from "Sapiens only, single template" -> full model.
steps = [
    ('Sapiens only\n(single template)',          68.1, 0.0),
    ('+ prompt ensemble',                          72.3, +4.2),
    ('+ MotionBERT (skeleton)',                    77.1, +4.8),  # ablation shows skel noise aug adds 2.7, ensemble model gives full 79.8
    ('+ skeleton noise aug.',                      79.8, +2.7),
]
labels = [s[0] for s in steps]
values = [s[1] for s in steps]
deltas = [s[2] for s in steps]

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(steps))
bars = ax.bar(x, values, color=['#7f7f7f', '#4bacc6', '#1f4e79', '#2ecc71'], alpha=0.9)
for i, (v, d) in enumerate(zip(values, deltas)):
    ax.text(i, v + 0.6, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')
    if d > 0:
        ax.text(i, v / 2, f'+{d:.1f} pp', ha='center', fontsize=9, color='white', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Top-1 accuracy on AzSLD 50 unseen (%)')
ax.set_ylim(60, 85)
ax.set_title('Component contributions (Table II)')
ax.text(0.99, -0.18, 'Source: paper Table II',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=8, color='#7f7f7f')

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_ablation_waterfall.pdf', bbox_inches='tight', dpi=300)
fig.savefig(FIG_DIR / 'fig_ablation_waterfall.png', bbox_inches='tight', dpi=300)
plt.show()

## Comparing your own results (if available)

If you have multiple checkpoints from different ablation runs, point this cell
at a list of `metrics.json` files to plot them side-by-side with the paper numbers.

In [ ]:
import json
ablation_runs = {
    # 'Full model': ROOT / 'outputs/exp_full/evaluation/metrics.json',
    # 'No skeleton': ROOT / 'outputs/exp_no_skeleton/evaluation/metrics.json',
}
found = []
for name, p in ablation_runs.items():
    if p.exists():
        m = json.loads(p.read_text())
        found.append((name, m.get('top1', float('nan')) * 100))

if found:
    names, tops = zip(*found)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(names, tops, color='#1f4e79', alpha=0.9)
    for i, v in enumerate(tops):
        ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontweight='bold')
    ax.set_xlabel('Top-1 (%)')
    ax.set_title('Your ablation runs')
    plt.tight_layout(); plt.show()
else:
    print('No ablation runs found. Add their metrics.json paths to `ablation_runs`.')